# PMSM simulation results

Plots from the C++ speed-control simulation. Regenerate the CSV with
`bazel run //:pmsm_sim` from `pmsmcpp`, then rerun this notebook.

Step plots show sampled or held signals. Changes between CSV logging ticks are visible only at the next recorded sample.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

csv_path = Path("pmsm_results.csv")
if not csv_path.is_file():
    csv_path = Path("pmsmcpp") / csv_path

results = pd.read_csv(csv_path)
time_ms = 1000 * results["time_s"]


def make_figure(rows, height):
    fig = make_subplots(rows=rows, cols=1, shared_xaxes=True, vertical_spacing=0.07)
    fig.update_layout(height=height, template="plotly_white", hovermode="x unified")
    fig.update_xaxes(title_text="Time (ms)", row=rows, col=1)
    return fig


def plot_signals(fig, row, columns, ylabel, *, held=False, group=None):
    for column, label, dash in columns:
        fig.add_trace(
            go.Scatter(
                x=time_ms,
                y=results[column],
                mode="lines",
                name=label,
                legendgroup=group,
                line=dict(dash=dash, shape="hv" if held else "linear"),
                hovertemplate="%{y:.5g}",
            ),
            row=row,
            col=1,
        )
    fig.update_yaxes(title_text=ylabel, row=row, col=1)

## Rotor motion

In [ ]:
fig = make_figure(rows=2, height=480)
plot_signals(fig, 1, [("omega_rad_s", "Speed", "solid")], "Speed (rad/s)")
plot_signals(fig, 2, [("theta_rad", "Position", "solid")], "Position (rad)")
fig.show()

## Phase currents and dq tracking

In [ ]:
fig = make_figure(rows=2, height=540)
plot_signals(
    fig, 1,
    [("ia_a", "Phase A", "solid"), ("ib_a", "Phase B", "solid"), ("ic_a", "Phase C", "solid")],
    "Phase current (A)",
    group="phase current",
)
plot_signals(
    fig, 2,
    [
        ("id_a", "d measured", "solid"),
        ("id_ref_a", "d reference", "dash"),
        ("iq_a", "q measured", "solid"),
        ("iq_ref_a", "q reference", "dash"),
    ],
    "dq current (A)",
    held=True,
    group="dq current",
)
fig.show()

## Voltage commands and inverter output

In [ ]:
fig = make_figure(rows=3, height=690)
plot_signals(
    fig, 1,
    [("vd_v", "d command", "solid"), ("vq_v", "q command", "solid")],
    "dq voltage (V)",
    held=True,
    group="dq voltage",
)
plot_signals(
    fig, 2,
    [("va_v", "Voltage A", "solid"), ("vb_v", "Voltage B", "solid"), ("vc_v", "Voltage C", "solid")],
    "Phase voltage (V)",
    held=True,
    group="phase voltage",
)
plot_signals(
    fig, 3,
    [("duty_a", "Duty A", "solid"), ("duty_b", "Duty B", "solid"), ("duty_c", "Duty C", "solid")],
    "Duty cycle",
    held=True,
    group="duty",
)
fig.update_yaxes(range=[-0.05, 1.05], row=3, col=1)
fig.show()